In [ ]:
training_time = eval_reallabor_utils.training_time(combined_eval_file, ['MRT', 'latent_model'])
training_time_errorbars = eval_reallabor_utils.bootstrap(eval_reallabor_utils.training_time, combined_eval_file, ['MRT', 'latent_model'], samples=10, relative_values=True, use_tqdm=True)
number_of_params = combined_eval_file.groupby('latent_model')['n_params'].mean()
number_of_params['InputsRegression'] = 8*15 + 15
number_of_params['KalmanFilter'] = 7*7 + 8*7 + 15*7
number_of_params['VAR1'] = 15*15 + 15*8 + 15

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
from plotting_styles import PaperStyle
from plotting_utils import bars, adjust_ylim

model_labels = {
                    'MovingAverage(1)': 'Last Step', 'MeanPredictor': 'Global Mean', 'InputsRegression': 'Linear Regression', 
                    'VAR1': 'VAR(1)', 'KalmanFilter': 'Kalman Filter', 
                    'clipped-shallow-PLRNN': 'PLRNN', 
                    # 'hierarchized-clipped-shallow-PLRNN': 'H-PLRNN',
                    'Transformer': 'Transformer'
                    }

with PaperStyle():

    fig, axes = plt.subplots(1, 2, figsize=(6.27, 3))

    bars(training_time.unstack('MRT').loc[model_labels.keys()].T.to_numpy(), ax=axes[0], 
         yerr=training_time_errorbars.unstack('MRT').loc[model_labels.keys()].T.to_numpy().reshape(2, 2, -1))      
    axes[0].set(ylabel='training time (seconds)', xticks=range(len(model_labels)))
    axes[0].set_xticklabels(model_labels.values(), rotation=45, ha='right')
    axes[0].legend(labels=['Sample 2', 'Sample 3'])
    adjust_ylim(axes[0], 0.03, 0)
    
    axes[1].bar(range(len(model_labels)), number_of_params.loc[model_labels.keys()])            
    axes[1].set(ylabel='# parameters', xticks=range(len(model_labels)))
    axes[1].set_xticklabels(model_labels.values(), rotation=45, ha='right')
    adjust_ylim(axes[1], 0.03, 0)

#     bars(success_ratio.unstack('MRT').loc[model_labels.keys()].T.to_numpy(), ax=axes[2])
#     axes[2].set(ylabel='convergence ratio', xticks=range(len(model_labels)))
#     axes[2].set_xticklabels(model_labels.values(), rotation=45, ha='right')
#     axes[2].set_ylim(0.5, 1)
#     axes[2].legend(labels=['MRT 2', 'MRT 3'])
#     adjust_ylim(axes[2], 0.03, 0)

    plt.tight_layout()
    if SAVE:
        plt.savefig('../results/_paper/training_time_number_of_params.png', dpi=300)
    plt.show()    